### Global Forest Recovery: Do Forest Carbon Storage changes corelate to  suggest recovery?

In [1]:
!pip install plotly country_converter -q

In [2]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import country_converter as coco
import logging

In [3]:
df = pd.read_csv("data/forest_and_carbon_data.csv")

## Proposition: Forest Loss Continues Despite Recovery Efforts

## Honest Interpretation
### Percent Change in Forest Extent (1992–2022)

In [ ]:
import json
from plotly.subplots import make_subplots
import plotly.graph_objects as go

logging.getLogger("country_converter").setLevel(logging.ERROR)
pio.renderers.default = "iframe_connected"

forest_df = df[df["INDICATOR"] == "Forest area, 1000 Hectares"].copy()
forest_df = forest_df.set_index("COUNTRY")
forest_df["1992"] = pd.to_numeric(forest_df["1992"], errors="coerce")
forest_df["2023"] = pd.to_numeric(forest_df["2022"], errors="coerce")

pct_change = ((forest_df["2022"] - forest_df["1992"]) / forest_df["1992"] * 100).dropna()
pct_change = pct_change.reset_index()
pct_change.columns = ["COUNTRY", "pct_change"]

pct_change["iso3"] = coco.convert(pct_change["COUNTRY"], to="ISO3", not_found=None)
pct_change = pct_change[pct_change["iso3"] != "not found"].dropna(subset=["iso3"])
pct_change = pct_change[pct_change["iso3"].apply(lambda x: isinstance(x, str))]

print(f"Countries mapped: {len(pct_change)}")

bins = [-float("inf"), -50, -20, -5, 5, 20, 50, float("inf")]
labels = [
    "Deep Loss (≤ -50%)", "Loss (-50% to -20%)", "Slight Loss (-20% to -5%)",
    "Stable (-5% to 5%)", "Slight Gain (5% to 20%)", "Gain (20% to 50%)", "Large Gain (≥ 50%)"
]

pct_change["category"] = pd.cut(pct_change["pct_change"], bins=bins, labels=labels)

# Global stats
_f1992  = pd.to_numeric(forest_df["1992"], errors="coerce")
_f2022  = pd.to_numeric(forest_df["2022"], errors="coerce")
_in_map = forest_df.index.isin(pct_change["COUNTRY"])
_g1992  = float(_f1992[_in_map].sum())
_g2022  = float(_f2022[_in_map].sum())
global_pct_val = round((_g2022 - _g1992) / _g1992 * 100, 2)
global_net_kha = round(_g2022 - _g1992, 0)

# Global yearly totals for scatter
_years = [str(y) for y in range(1992, 2023)]
_global_yearly = (
    forest_df[_in_map][_years]
    .apply(pd.to_numeric, errors="coerce")
    .sum()
    .reset_index()
)
_global_yearly.columns = ["Year", "ForestArea_kha"]
_global_yearly["Year"] = _global_yearly["Year"].astype(int)
_global_yearly["ForestArea_Mha"] = _global_yearly["ForestArea_kha"] / 1000
_global_yearly_vals = _global_yearly["ForestArea_Mha"].round(2).tolist()

# Per-country yearly data
_country_yearly = {}
for _, _r in pct_change.iterrows():
    _c, _iso3 = _r["COUNTRY"], _r["iso3"]
    if _c not in forest_df.index:
        continue
    _vals = pd.to_numeric(forest_df.loc[[_c], _years].iloc[0], errors="coerce")
    _country_yearly[_iso3] = [
        None if pd.isna(v) else round(float(v) / 1000, 2) for v in _vals
    ]

# ── Recovery deficit stats ─────────────────────────────────────────────────────
_in_loss = forest_df.index.isin(pct_change.loc[pct_change["pct_change"] < 0, "COUNTRY"])
_in_gain = forest_df.index.isin(pct_change.loc[pct_change["pct_change"] > 0, "COUNTRY"])

_total_loss_mha   = round((_f1992[_in_loss].sum() - _f2022[_in_loss].sum()) / 1000, 1)
_total_gain_mha   = round((_f2022[_in_gain].sum() - _f1992[_in_gain].sum()) / 1000, 1)
_net_deficit_mha  = round(_total_loss_mha - _total_gain_mha, 1)
_mha_per_year     = round(_net_deficit_mha / 4, 1)
_loss_count       = int((pct_change["pct_change"] < 0).sum())
_gain_count       = int((pct_change["pct_change"] > 0).sum())
_annual_gain_rate = round(_total_gain_mha / 30, 1)
_rate_multiplier  = round(_mha_per_year / _annual_gain_rate, 1) if _annual_gain_rate > 0 else 0

deficit_stats = {
    "loss_mha":     _total_loss_mha,
    "gain_mha":     _total_gain_mha,
    "deficit_mha":  _net_deficit_mha,
    "mha_per_year": _mha_per_year,
    "loss_count":   _loss_count,
    "gain_count":   _gain_count,
    "multiplier":   _rate_multiplier,
}
print(f"Lost: {_total_loss_mha} Mha | Gained: {_total_gain_mha} Mha | Deficit: {_net_deficit_mha} Mha | Need: {_mha_per_year} Mha/yr | {_rate_multiplier}x current rate")

color_map = {
    "Deep Loss (≤ -50%)":        "#8c510a",
    "Loss (-50% to -20%)":       "#bf812d",
    "Slight Loss (-20% to -5%)": "#dfc27d",
    "Stable (-5% to 5%)":        "#ebf1e7",
    "Slight Gain (5% to 20%)":   "#80cdc1",
    "Gain (20% to 50%)":         "#35978f",
    "Large Gain (≥ 50%)":        "#01665e",
}

_cho = px.choropleth(
    pct_change,
    locations="iso3",
    color="category",
    hover_name="COUNTRY",
    hover_data={"pct_change": ":.1f", "category": False, "iso3": False},
    category_orders={"category": labels},
    color_discrete_map=color_map,
    labels={"category": "% Change in Forest Area"}
)

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.82, 0.18],
    specs=[[{"type": "choropleth"}], [{"type": "scatter"}]],
    vertical_spacing=0.005,
    subplot_titles=("", "Global Time Series Projection"),
)

for trace in _cho.data:
    fig.add_trace(trace, row=1, col=1)

fig.add_trace(
    go.Scatter(
        x=_global_yearly["Year"],
        y=_global_yearly["ForestArea_Mha"],
        mode="lines+markers",
        line=dict(color="#8c510a", width=2.5),
        marker=dict(color="#8c510a", size=5),
        hovertemplate="%{x}: %{y:.0f} Mha<extra></extra>",
        showlegend=False,
    ),
    row=2, col=1,
)

fig.update_geos(
    showcoastlines=True,
    showframe=False,
    showcountries=True,
    landcolor="#d3d3d3",
    fitbounds=False,
)

fig.update_layout(
    width=955,
    height=660,
    title_text="Rooted in Reality: Restoration Efforts Are Outpaced by Forest Loss<br><sup>Percent change in forest area, 1992–2022 | Source: FAO | Glasgow Leaders&#39; Declaration (COP26, 2021)</sup>",
    title_font_size=20,
    title_x=0.50,
    title_y=0.97,
    legend_title_text="% Change in Forest Area",
    legend=dict(x=1.0, y=0.85, xanchor="left", yanchor="middle"),
    margin=dict(l=0, r=200, t=60, b=60),
    uirevision="constant",
    xaxis2=dict(title="Year", tickmode="linear", dtick=5),
    yaxis2=dict(title="Forest Area (Mha)"),
)

click_js = f"""
(function() {{
    var gd = document.getElementsByClassName('plotly-graph-div')[0];
    if (!gd) return;

    var colorMap      = {json.dumps(color_map)};
    var globalStats   = {json.dumps({"pct": global_pct_val, "net_kha": global_net_kha})};
    var countryYearly = {json.dumps(_country_yearly)};
    var globalYearly  = {json.dumps(_global_yearly_vals)};
    var deficitStats  = {json.dumps(deficit_stats)};

    var origSnaps = gd.data.map(function(t) {{
        if (t.type !== 'choropleth') return null;
        return {{
            locations:  (t.locations || []).slice(),
            text:       t.text       ? t.text.slice() : null,
            customdata: t.customdata ? t.customdata.map(function(r) {{
                return Array.isArray(r) ? r.slice() : r;
            }}) : null,
        }};
    }});

    var iso3Map = {{}};
    origSnaps.forEach(function(snap, ti) {{
        if (!snap) return;
        snap.locations.forEach(function(iso3, pi) {{ iso3Map[iso3] = {{ti: ti, pi: pi}}; }});
    }});

    gd.style.position = 'relative';

    // ── Side stat cards (below legend, right margin) ───────────────────────────
    var d = deficitStats;
    var sideCardDefs = [
        {{ label: 'Forest Lost',    value: d.loss_mha.toFixed(1) + ' Mha',        sub: d.loss_count + ' countries',  accent: '#8c510a' }},
        {{ label: 'Forest Gained',  value: d.gain_mha.toFixed(1) + ' Mha',        sub: d.gain_count + ' countries',  accent: '#35978f' }},
        {{ label: 'Net Deficit',    value: d.deficit_mha.toFixed(1) + ' Mha',     sub: 'unrecovered loss',            accent: '#f97316' }},
        {{ label: 'Needed by 2030', value: d.mha_per_year.toFixed(1) + ' Mha/yr', sub: 'to break even',              accent: '#f97316' }},
    ];

    var sideCards = document.createElement('div');
    sideCards.style.cssText = 'position:absolute;left:762px;top:230px;width:183px;font-family:Arial,sans-serif;pointer-events:none';
    sideCards.innerHTML = sideCardDefs.map(function(s) {{
        return '<div style="border-left:3px solid ' + s.accent + ';padding:6px 10px;margin-bottom:7px;background:#fafafa;border-radius:0 3px 3px 0">' +
            '<div style="font-size:8px;letter-spacing:1.4px;text-transform:uppercase;color:' + s.accent + ';margin-bottom:2px">' + s.label + '</div>' +
            '<div style="font-size:16px;font-weight:bold;color:#1a1a1a;line-height:1.1;margin-bottom:1px">' + s.value + '</div>' +
            '<div style="font-size:10px;color:#888">' + s.sub + '</div>' +
            '</div>';
    }}).join('');
    gd.appendChild(sideCards);

    // ── Recovery deficit callout (title + narrative only) ─────────────────────
    var narrative =
        'Net-loss countries have shed <strong style="color:#fff">' + d.loss_mha.toFixed(1) + ' Mha</strong> of forest since 1992 ' +
        'across <strong style="color:#fff">' + d.loss_count + ' countries</strong>. ' +
        'Gains across <strong style="color:#fff">' + d.gain_count + ' countries</strong> have recovered only ' +
        '<strong style="color:#fff">' + d.gain_mha.toFixed(1) + ' Mha</strong> — leaving a deficit of ' +
        '<strong style="color:#f97316">' + d.deficit_mha.toFixed(1) + ' Mha</strong>. ' +
        'Closing that gap by 2030 requires <strong style="color:#f97316">' + d.mha_per_year.toFixed(1) + ' Mha of restoration per year</strong> — ' +
        '<strong style="color:#f97316">' + d.multiplier.toFixed(1) + '× the annual rate achieved over the last 30 years.</strong>';

    var callout = document.createElement('div');
    callout.style.cssText = 'width:955px;background:#1c1c2e;color:#e8e8f0;border-radius:8px;padding:20px 28px 22px;margin-top:10px;font-family:Arial,sans-serif;box-sizing:border-box';
    callout.innerHTML =
        '<div style="font-size:10px;letter-spacing:2px;text-transform:uppercase;color:#f97316;margin-bottom:10px">The Recovery Deficit</div>' +
        '<div style="font-size:13px;color:#b0b0c8;line-height:1.7">' + narrative + '</div>';
    gd.insertAdjacentElement('afterend', callout);

    // ── Stats box (aligned with time series subplot) ───────────────────────────
    var statsBox = document.createElement('div');
    statsBox.style.cssText = [
        'position:absolute',
        'left:762px',
        'top:505px',
        'width:183px',
        'background:#fff',
        'border:1px solid #ccc',
        'border-radius:5px',
        'padding:10px 12px',
        'font-family:Arial,sans-serif',
        'font-size:12px',
        'line-height:1.6',
        'box-shadow:0 1px 5px rgba(0,0,0,0.12)',
        'pointer-events:none',
    ].join(';');
    gd.appendChild(statsBox);

    function pctToTextColor(pct) {{
        var c;
        if      (pct <= -50) c = colorMap['Deep Loss (≤ -50%)'];
        else if (pct <= -20) c = colorMap['Loss (-50% to -20%)'];
        else if (pct <=  -5) c = colorMap['Slight Loss (-20% to -5%)'];
        else if (pct <=   5) c = colorMap['Stable (-5% to 5%)'];
        else if (pct <=  20) c = colorMap['Slight Gain (5% to 20%)'];
        else if (pct <=  50) c = colorMap['Gain (20% to 50%)'];
        else                  c = colorMap['Large Gain (≥ 50%)'];
        return (c === '#ebf1e7' || c === '#ffffff') ? '#888' : c;
    }}

    function showGlobalStats() {{
        var pctColor = pctToTextColor(globalStats.pct);
        var sign     = globalStats.pct >= 0 ? '+' : '';
        var netMha   = globalStats.net_kha / 1000;
        var netSign  = netMha >= 0 ? '+' : '';
        statsBox.innerHTML =
            '<div style="font-weight:bold;font-size:13px;border-bottom:1px solid #eee;' +
            'padding-bottom:5px;margin-bottom:6px">Global (1992–2022)</div>' +
            '<div style="margin-bottom:3px">Change: <strong style="color:' + pctColor + '">' +
            sign + globalStats.pct.toFixed(1) + '%</strong></div>' +
            '<div>Net: <strong style="color:' + pctColor + '">' +
            netSign + netMha.toFixed(1) + ' Mha</strong></div>';
    }}

    function showCountryStats(name, pctChange, category) {{
        var color  = colorMap[category] || '#ccc';
        var border = color === '#ffffff' || color === '#ebf1e7' ? '#aaa' : color;
        var sign   = pctChange >= 0 ? '+' : '';
        statsBox.innerHTML =
            '<div style="font-weight:bold;font-size:13px;border-bottom:1px solid #eee;' +
            'padding-bottom:5px;margin-bottom:6px">' + name + '</div>' +
            '<div style="margin-bottom:4px">Change: <strong>' + sign + pctChange.toFixed(1) + '%</strong></div>' +
            '<div style="display:flex;align-items:center">' +
            '<div style="width:11px;height:11px;background:' + color + ';border:1px solid ' + border + ';' +
            'margin-right:7px;flex-shrink:0;border-radius:2px"></div>' +
            '<span style="color:#444">' + category + '</span>' +
            '</div>';
    }}

    showGlobalStats();

    var selected = null;

    function applySelection(selectedTi, selectedPi, newScatterY, newTitle) {{
        var newData = gd.data.map(function(t, i) {{
            if (t.type !== 'choropleth') {{
                if (newScatterY !== undefined) return Object.assign({{}}, t, {{y: newScatterY}});
                return t;
            }}
            var isActive   = selectedTi !== null;
            var isSelected = isActive && (i === selectedTi);
            return Object.assign({{}}, t, {{
                selectedpoints: isActive ? (isSelected ? [selectedPi] : []) : null,
                selected:       {{marker: {{opacity: 1}}}},
                unselected:     {{marker: {{opacity: isActive ? 0.15 : 1}}}},
            }});
        }});
        if (newTitle !== undefined && gd.layout.annotations && gd.layout.annotations.length >= 1) {{
            gd.layout.annotations[0].text = newTitle;
        }}
        Plotly.react(gd, newData, gd.layout);
    }}

    function reset() {{
        applySelection(null, null, globalYearly, 'Global Time Series Projection');
        showGlobalStats();
        selected = null;
    }}

    function selectCountry(iso3) {{
        var info = iso3Map[iso3];
        if (!info) return;

        var snap     = origSnaps[info.ti];
        var name     = snap.text       ? snap.text[info.pi]                        : iso3;
        var cdRow    = snap.customdata ? snap.customdata[info.pi]                  : null;
        var pctVal   = cdRow !== null  ? (Array.isArray(cdRow) ? cdRow[0] : cdRow) : null;
        var category = gd.data[info.ti].name;

        var countryY = countryYearly.hasOwnProperty(iso3) ? countryYearly[iso3] : null;
        var title    = countryY !== null ? name + ' Time Series Projection' : 'Global Time Series Projection';
        applySelection(info.ti, info.pi, countryY !== null ? countryY : globalYearly, title);
        if (pctVal !== null) showCountryStats(name, pctVal, category);
        selected = iso3;
    }}

    gd.on('plotly_click', function(data) {{
        var iso3 = (data.points && data.points.length) ? data.points[0].location : null;
        if (!iso3 || iso3 === selected) reset();
        else selectCountry(iso3);
    }});

    gd.addEventListener('click', function(e) {{
        if (selected === null) return;
        var el = e.target;
        while (el && el !== gd) {{
            if (el.classList && el.classList.contains('choropleth')) return;
            el = el.parentNode;
        }}
        reset();
    }});
}})();
"""

fig.write_html("forest_loss_map.html", post_script=click_js)
print("Saved: forest_loss_map.html")
fig.show()


In [170]:
total_1992 = forest_df["1992"].sum()
total_2022 = forest_df["2022"].sum()

pct_global = (total_2022 - total_1992) / total_1992 * 100

print("Global Forest Area:", pct_global)

Global Forest Area: -2.5479275043177023


In [171]:
net_change = (forest_df["2022"] - forest_df["1992"]).sum()

In [172]:
print("Net Change in Forest Area:", net_change)

Net Change in Forest Area: -542575.5896015903


## Dishonest Interpretation
### Percent Change in Forest Extent (1992–2022)

In [173]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import country_converter as coco
import logging

logging.getLogger("country_converter").setLevel(logging.ERROR)
pio.renderers.default = "iframe_connected"

forest_df = df[df["INDICATOR"] == "Forest area, 1000 Hectares"].copy()
forest_df = forest_df.set_index("COUNTRY")
forest_df["1992"] = pd.to_numeric(forest_df["1992"], errors="coerce")
forest_df["2022"] = pd.to_numeric(forest_df["2022"], errors="coerce")

pct_change = ((forest_df["2022"] - forest_df["1992"]) / forest_df["1992"] * 100).dropna()
pct_change = pct_change.reset_index()
pct_change.columns = ["COUNTRY", "pct_change"]

pct_change["iso3"] = coco.convert(pct_change["COUNTRY"], to="ISO3", not_found=None)
pct_change = pct_change[pct_change["iso3"] != "not found"].dropna(subset=["iso3"])

countries_gaining = (pct_change["pct_change"] > 0).sum()
countries_losing = (pct_change["pct_change"] < 0).sum()

print(f"Countries mapped: {len(pct_change)}")
print(f"Countries Increasing Forest Area: {countries_gaining}")

bins = [-float("inf"), -0.1, 0.1, float("inf")]
labels = [
    "Loss", 
    "Stable", 
    "Gain"
]

pct_change["category"] = pd.cut(pct_change["pct_change"], bins=bins, labels=labels)

color_map = {
    "Loss":        "#efe6d7",
    "Stable":        "#ffffff",
    "Gain":        "#00b140",
}

fig = px.choropleth(
    pct_change,
    locations="iso3",
    color="category",
    hover_name="COUNTRY",
    hover_data={"pct_change": ":.1f", "category": False, "iso3": False},
    category_orders={"category": labels},
    color_discrete_map=color_map,
    title="Global Forest Recovery is Underway<br><sup>Forest Gains and Losses, 1992–2023</sup>",
    labels={"category": "Gains and losses"}
)

fig.update_geos(
    showcoastlines=True, 
    showframe=False, 
    showcountries=True, 
    landcolor="#d0d0d0",
    domain=dict(y=[0.00, 0.99])
)
    
fig.update_layout(
    width=955, 
    height=540,
    title_font_size=20,
    title_x=0.50,
    title_y=0.80,
    legend_title_text="% Change in Forest Area",
    legend=dict(x=1.0, y=0.7, xanchor="left", yanchor="middle"),
    margin=dict(l=0, r=200, t=75, b=0)
)

fig.write_html("forest_gain_map.html")
print("Saved: forest_gain_map.html")

fig.show()

Countries mapped: 233
Countries Increasing Forest Area: 95
Saved: forest_gain_map.html


In [174]:
pct_change.nlargest(1, "pct_change")

,COUNTRY,pct_change,iso3,category
159,"Bahrain, Kingdom of",196.0,BHR,Gain


## Honest #2

In [178]:
forest_country_df = forest_df[
    ~forest_df.index.isin(exclude)
].copy()

years = [str(y) for y in range(1992, 2023)]

global_forest = (
    forest_country_df[years]
    .apply(pd.to_numeric, errors="coerce")
    .sum()
)

global_forest = global_forest.reset_index()
global_forest.columns = ["Year", "ForestArea"]

global_forest["Year"] = global_forest["Year"].astype(int)

fig = px.line(
    global_forest,
    x="Year",
    y="ForestArea",
    title="Forest Loss Continues Despite Recovery Efforts"
)

fig.update_traces(
    line=dict(
        color="#8c510a",
        width=4
    )
)

fig.update_layout(
    width=1000,
    height=600,
    xaxis_title="Year",
    yaxis_title="Forest Area (1000 Hectares)",
    showlegend=False
)
fig.write_html("forest_reduction_line.html")
print("Saved: forest_reduction_line.html")
fig.show()

Saved: forest_reduction_line.html


## Dishonest #2
#### Forest recovery efforts have produced substantial gains in a number of countries, resulting in a long positive tail in the distribution.

In [176]:
forest_df["change_pct"] = (
    (forest_df["2022"] - forest_df["1992"])
    / forest_df["1992"]
    * 100
)

all_df = (
    forest_df[["change_pct"]]
    .dropna()
    .reset_index()
    .sort_values("change_pct", ascending=False)
)

all_df["iso3"] = coco.convert(
    all_df["COUNTRY"],
    to="ISO3",
    not_found=None
)

all_df = all_df[
    (all_df["iso3"].notna()) &
    (all_df["iso3"] != "not found")
]

exclude = [
    "G7",
    "G20",
    "Asia",
    "Africa",
    "Americas",
    "Euro Area (EA) - aggregate of member states",
    "World",
    "European Union",
    "Advanced Economies",
    "Emerging Market and Developing Economies"
]

all_df = all_df[
    ~all_df["COUNTRY"].isin(exclude)
]

In [179]:
fig = px.histogram(
    all_df,
    x="change_pct",
    nbins=40,
    title="Forest Gain By Country: Distribution of Forest Area Change by Country"
)

fig.add_vline(x=0, line_dash="dash")
fig.write_html("forest_gain_dist.html")
print("Saved: forest_gain_dist.html")
fig.show()

Saved: forest_gain_dist.html


## Proposition: El Niño Events Are Associated with More Rain-Related Disasters

In [166]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe_connected"

# Rain-related disaster indicators (flood + landslide/mudslide)
FLOOD_KEY     = "Climate related disasters frequency, Number of Disasters: Flood, Number"
LANDSLIDE_KEY = "Climate related disasters frequency, Number of Disasters: Landslide, Number"

years       = list(range(1960, 2025))
annual_cols = [str(y) for y in years]

df_raw    = pd.read_csv("data/forest_and_carbon_data.csv",
                        usecols=["INDICATOR", "COUNTRY"] + annual_cols)
rain_df   = df_raw[df_raw["INDICATOR"].isin([FLOOD_KEY, LANDSLIDE_KEY])]
sums      = rain_df.groupby("INDICATOR")[annual_cols].sum(numeric_only=True)

annual = pd.DataFrame({
    "year":      years,
    "flood":     sums.loc[FLOOD_KEY].values,
    "landslide": sums.loc[LANDSLIDE_KEY].values,
})
annual["rain_disasters"] = annual["flood"] + annual["landslide"]

# El Niño year classification (NOAA ONI ≥ +0.5 °C, both onset and peak year included)
el_nino_years = {
    1965, 1966, 1972, 1973, 1976, 1977,
    1982, 1983, 1987, 1988, 1991, 1992, 1994, 1995,
    1997, 1998, 2002, 2003, 2004, 2005, 2006, 2007,
    2009, 2010, 2015, 2016, 2018, 2019, 2023, 2024,
}

# Major events to highlight (strongest El Niño episodes)
major_events = [(1982, 1983), (1997, 1998), (2015, 2016), (2023, 2024)]

annual["enso"] = annual["year"].map(
    lambda y: "El Niño" if y in el_nino_years else "Neutral / La Niña"
)

# ── Time series ────────────────────────────────────────────────────────────────
fig_ts = go.Figure()

# Shade each major El Niño window
for start, end in major_events:
    peak = annual.loc[annual["year"].between(start, end), "rain_disasters"].max()
    fig_ts.add_vrect(
        x0=start - 0.5, x1=end + 0.5,
        fillcolor="#f97316", opacity=0.20, line_width=0,
    )
    fig_ts.add_annotation(
        x=(start + end) / 2, y=peak + 16,
        text=f"<b>{start}–{str(end)[-2:]}</b>",
        showarrow=False,
        font=dict(color="#c2410c", size=10),
    )

# Main line, markers colored by ENSO phase
fig_ts.add_trace(go.Scatter(
    x=annual["year"],
    y=annual["rain_disasters"],
    mode="lines+markers",
    line=dict(color="#1e40af", width=2),
    marker=dict(
        size=6,
        color=annual["enso"].map({"El Niño": "#f97316", "Neutral / La Niña": "#1e40af"}),
        line=dict(width=0.5, color="white"),
    ),
    hovertemplate="%{x}: %{y} disasters<extra></extra>",
    showlegend=False,
))

# Legend entries
for label, color in [("El Niño year", "#f97316"), ("Neutral / La Niña year", "#1e40af")]:
    fig_ts.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(color=color, size=9), name=label,
    ))

fig_ts.update_layout(
    title="Rain-Related Disasters Over Time<br>"
          "<sup>Global floods + landslides/mudslides, 1960–2024 | Shaded = major El Niño events</sup>",
    xaxis_title="Year",
    yaxis_title="Number of Disasters",
    width=955, height=480,
    title_font_size=18,
    title_x=0.5,
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.7)"),
    margin=dict(l=60, r=40, t=100, b=60),
)

fig_ts.write_html("el_nino_timeseries.html")
print("Saved: el_nino_timeseries.html")
fig_ts.show()

Saved: el_nino_timeseries.html


In [167]:
import plotly.express as px

avg = (
    annual
    .groupby("enso")["rain_disasters"]
    .agg(mean="mean", std="std")
    .reset_index()
    .rename(columns={"enso": "ENSO Phase"})
)

fig_bar = px.bar(
    avg,
    x="ENSO Phase",
    y="mean",
    color="ENSO Phase",
    color_discrete_map={"El Niño": "#f97316", "Neutral / La Niña": "#1e40af"},
    error_y="std",
    title="El Niño Years Average More Rain-Related Disasters<br>"
          "<sup>Mean flood + landslide events per year, 1960–2024</sup>",
    labels={"mean": "Avg Disasters / Year", "ENSO Phase": ""},
    text="mean",
)
fig_bar.update_traces(texttemplate="%{text:.0f}", textposition="outside")
fig_bar.update_layout(
    width=620,
    height=480,
    title_font_size=18,
    title_x=0.5,
    showlegend=False,
    yaxis_title="Avg Disasters / Year",
    margin=dict(l=60, r=40, t=100, b=80),
)

fig_bar.write_html("el_nino_bar.html")
print("Saved: el_nino_bar.html")
fig_bar.show()

Saved: el_nino_bar.html


In [168]:

# ── Regional exploratory analysis: does El Niño redistribute flood risk? ──────
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe_connected"

AFFECTED_KEY = "Climate related disasters frequency, Number of People Affected: Flood"

years_exp   = list(range(1980, 2025))
annual_exp  = [str(y) for y in years_exp]

df_exp  = pd.read_csv("data/forest_and_carbon_data.csv",
                      usecols=["INDICATOR", "COUNTRY"] + annual_exp)
aff_raw = df_exp[df_exp["INDICATOR"] == AFFECTED_KEY].set_index("COUNTRY")[annual_exp]
aff_raw = aff_raw.apply(pd.to_numeric, errors="coerce")

el_nino_years = {
    1982, 1983, 1987, 1988, 1991, 1992, 1994, 1995,
    1997, 1998, 2002, 2003, 2004, 2005, 2006, 2007,
    2009, 2010, 2015, 2016, 2018, 2019, 2023, 2024,
}
nino_cols    = [str(y) for y in years_exp if y in el_nino_years]
neutral_cols = [str(y) for y in years_exp if y not in el_nino_years]

aff_raw["el_nino_avg"]    = aff_raw[nino_cols].mean(axis=1)
aff_raw["neutral_avg"]    = aff_raw[neutral_cols].mean(axis=1)
aff_raw["ratio"]          = (aff_raw["el_nino_avg"] / aff_raw["neutral_avg"].replace(0, float("nan"))).round(2)

regions = {
    "East Africa":     ["Kenya", "Somalia", "Uganda", "Ethiopia", "Tanzania"],
    "Southeast Asia":  ["Indonesia", "Philippines", "Thailand", "Vietnam"],
    "South America":   ["Peru", "Ecuador", "Chile", "Bolivia", "Colombia"],
}

rows = []
for region, countries in regions.items():
    for c in countries:
        if c in aff_raw.index:
            rows.append({
                "region":      region,
                "country":     c,
                "El Niño avg": aff_raw.loc[c, "el_nino_avg"] / 1e6,
                "Neutral avg": aff_raw.loc[c, "neutral_avg"] / 1e6,
                "ratio":       aff_raw.loc[c, "ratio"],
            })

region_df = pd.DataFrame(rows)
print(region_df.to_string(index=False))

# Grouped bar: El Niño avg vs Neutral avg per country, faceted by region
fig = go.Figure()

fig.add_trace(go.Bar(
    name="El Niño avg",
    x=[f"{r['region']}<br>{r['country']}" for _, r in region_df.iterrows()],
    y=region_df["El Niño avg"],
    marker_color="#f97316",
))
fig.add_trace(go.Bar(
    name="Neutral avg",
    x=[f"{r['region']}<br>{r['country']}" for _, r in region_df.iterrows()],
    y=region_df["Neutral avg"],
    marker_color="#1e40af",
))

fig.update_layout(
    barmode="group",
    title="People Affected by Floods: El Niño vs Neutral Years by Region<br>"
          "<sup>Avg people affected per year (millions), 1980–2024</sup>",
    yaxis_title="Avg People Affected (millions)",
    xaxis_title="",
    width=955, height=500,
    title_font_size=16,
    title_x=0.5,
    legend=dict(x=0.78, y=0.99),
    margin=dict(l=60, r=40, t=100, b=80),
)

fig.show()


        region     country  El Niño avg  Neutral avg  ratio
   East Africa       Kenya     0.249629     0.188473   1.32
   East Africa     Somalia     0.438975     0.179810   2.44
   East Africa      Uganda     0.097810     0.051478   1.90
Southeast Asia   Indonesia     0.260331     0.317420   0.82
Southeast Asia Philippines     0.731932     1.054693   0.69
Southeast Asia    Thailand     1.326886     1.893536   0.70
Southeast Asia     Vietnam     0.440642     1.160303   0.38
 South America        Peru     0.081340     0.269061   0.30
 South America     Ecuador     0.102637     0.048384   2.12
 South America       Chile     0.067214     0.038506   1.75
 South America     Bolivia     0.136161     0.159826   0.85
 South America    Colombia     0.390783     0.298521   1.31
